# 🏪 Notebook 3: Redis as Cache vs Primary Store

Redis is most commonly used as a **cache**, but it can also serve as a **primary data store** for certain use cases. In this notebook, we'll implement caching patterns, explore TTL and eviction, and build features where Redis is the source of truth — distributed locks, rate limiters, and leaderboards.

## Learning Objectives
- Implement the Cache-Aside pattern with Redis
- Understand TTL, eviction policies, and cache invalidation
- Implement the Write-Through pattern
- Build a distributed lock with Redis
- Build a rate limiter with Redis
- Understand when Redis can be the primary store vs just a cache

## 🛠️ Setup

```bash
cd 03-technologies/databases/redis
docker compose up -d
```

### Visualization
- **RedisInsight**: http://localhost:5540 — connect to `redis://localhost:6379`

### Kernel Selection
Select the `.venv` kernel in VS Code's kernel picker (top-right).
If it doesn't appear, reload: `Cmd+Shift+P` → "Reload Window".

In [1]:
import redis
import json
import time
import random

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

try:
    r.ping()
    print("✅ Connected to Redis")
    r.flushdb()
    print("🧹 Flushed database for a clean start")
except Exception as e:
    print(f"❌ Redis connection failed: {e}")
    print("   Run: cd 03-technologies/databases/redis && docker compose up -d")

# Simulate a "database" using a Python dictionary
# (In a real app, this would be PostgreSQL, MySQL, etc.)
FAKE_DB = {
    "product:1": {"id": 1, "name": "Laptop", "price": 999.99, "stock": 50},
    "product:2": {"id": 2, "name": "Mouse", "price": 29.99, "stock": 200},
    "product:3": {"id": 3, "name": "Keyboard", "price": 79.99, "stock": 150},
    "product:4": {"id": 4, "name": "Monitor", "price": 449.99, "stock": 30},
    "product:5": {"id": 5, "name": "Headphones", "price": 149.99, "stock": 75},
}

def db_read(key):
    """Simulate a slow database read (10ms latency)."""
    time.sleep(0.01)  # Simulated disk I/O latency
    return FAKE_DB.get(key)

def db_write(key, value):
    """Simulate a database write."""
    time.sleep(0.01)
    FAKE_DB[key] = value

print(f"📦 Fake database loaded with {len(FAKE_DB)} products")

✅ Connected to Redis
🧹 Flushed database for a clean start
📦 Fake database loaded with 5 products


## 1️⃣ Cache-Aside Pattern

The most common caching pattern. The application checks the cache first. On a **miss**, it reads from the database and stores the result in the cache.

```
Client → Cache hit?
         ├── YES → Return cached data  ⚡ Fast!
         └── NO  → Read from DB → Store in cache → Return data
```

This is the default pattern for most applications. It's simple, widely understood, and works well.

In [2]:
def get_product_cache_aside(product_id):
    """Cache-Aside: check cache first, fall back to DB on miss."""
    cache_key = f"product:{product_id}"
    
    # Step 1: Check the cache
    cached = r.get(cache_key)
    if cached:
        print(f"  ⚡ Cache HIT for {cache_key}")
        return json.loads(cached)
    
    # Step 2: Cache miss — read from database
    print(f"  💾 Cache MISS for {cache_key} — reading from DB...")
    data = db_read(cache_key)
    
    if data is None:
        return None
    
    # Step 3: Store in cache with a TTL (time to live)
    r.set(cache_key, json.dumps(data), ex=60)  # Cache for 60 seconds
    print(f"  📝 Stored in cache (TTL=60s)")
    
    return data

# First call: cache miss
print("Request 1:")
product = get_product_cache_aside(1)
print(f"  Result: {product}\n")

# Second call: cache hit
print("Request 2:")
product = get_product_cache_aside(1)
print(f"  Result: {product}\n")

# Different product: cache miss
print("Request 3 (different product):")
product = get_product_cache_aside(2)
print(f"  Result: {product}")

print("\n💡 First request is slow (DB read), subsequent requests are fast (cache hit)!")

Request 1:
  💾 Cache MISS for product:1 — reading from DB...
  📝 Stored in cache (TTL=60s)
  Result: {'id': 1, 'name': 'Laptop', 'price': 999.99, 'stock': 50}

Request 2:
  ⚡ Cache HIT for product:1
  Result: {'id': 1, 'name': 'Laptop', 'price': 999.99, 'stock': 50}

Request 3 (different product):
  💾 Cache MISS for product:2 — reading from DB...
  📝 Stored in cache (TTL=60s)
  Result: {'id': 2, 'name': 'Mouse', 'price': 29.99, 'stock': 200}

💡 First request is slow (DB read), subsequent requests are fast (cache hit)!


In [3]:
# Let's measure the actual performance difference

# Warm up the cache
for pid in range(1, 6):
    get_product_cache_aside(pid)
print("Cache warmed up!\n")

# Measure cache hits vs misses
def benchmark(label, fetch_fn, product_ids, iterations=50):
    times = []
    for _ in range(iterations):
        pid = random.choice(product_ids)
        start = time.time()
        fetch_fn(pid)
        elapsed = (time.time() - start) * 1000
        times.append(elapsed)
    avg = sum(times) / len(times)
    print(f"{label}: avg={avg:.2f}ms, min={min(times):.2f}ms, max={max(times):.2f}ms")

# All cache hits (data is cached)
benchmark("⚡ Cache hits  ", get_product_cache_aside, [1, 2, 3, 4, 5])

# Force cache misses by flushing
r.flushdb()
benchmark("💾 Cache misses", get_product_cache_aside, [1, 2, 3, 4, 5])

print("\n💡 Cache hits are significantly faster because they skip the database!")

  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:2
  💾 Cache MISS for product:3 — reading from DB...
  📝 Stored in cache (TTL=60s)
  💾 Cache MISS for product:4 — reading from DB...
  📝 Stored in cache (TTL=60s)
  💾 Cache MISS for product:5 — reading from DB...


  📝 Stored in cache (TTL=60s)
Cache warmed up!

  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:3
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:3
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:3
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:3
  ⚡ Cache HIT for product:4


  📝 Stored in cache (TTL=60s)
  💾 Cache MISS for product:4 — reading from DB...
  📝 Stored in cache (TTL=60s)
  💾 Cache MISS for product:2 — reading from DB...
  📝 Stored in cache (TTL=60s)
  💾 Cache MISS for product:5 — reading from DB...


  📝 Stored in cache (TTL=60s)
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:2
  💾 Cache MISS for product:3 — reading from DB...


  📝 Stored in cache (TTL=60s)
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:3
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:3
  ⚡ Cache HIT for product:3
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:2
  ⚡ Cache HIT for product:3
  ⚡ Cache HIT for product:3
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:3
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:1
  ⚡ Cache HIT for product:5
  ⚡ Cache HIT for product:4
  ⚡ Cache HIT for 

## 2️⃣ TTL and Eviction

**TTL (Time To Live)** automatically expires keys after a set time. This keeps the cache fresh and prevents it from growing unbounded.

**Eviction policies** determine what happens when Redis runs out of memory:

| Policy | Behavior | Best For |
|--------|----------|----------|
| `noeviction` | Returns error when full | Primary data store |
| `allkeys-lru` | Evicts least recently used | General-purpose cache |
| `volatile-lru` | Evicts LRU keys with TTL set | Mixed cache + persistent |
| `allkeys-random` | Evicts random keys | When all keys are equal |
| `volatile-ttl` | Evicts keys closest to expiring | Time-sensitive cache |

In [4]:
r.flushdb()

# Set a key with a 5-second TTL
r.set("flash_sale:item42", json.dumps({"discount": "50%", "item": "Laptop"}), ex=5)

print("Key set with 5-second TTL")
print(f"  TTL remaining: {r.ttl('flash_sale:item42')} seconds")
print(f"  Value: {r.get('flash_sale:item42')}")

# Wait and check again
time.sleep(2)
print(f"\nAfter 2 seconds:")
print(f"  TTL remaining: {r.ttl('flash_sale:item42')} seconds")
print(f"  Value: {r.get('flash_sale:item42')}")

# Wait for expiration
time.sleep(4)
print(f"\nAfter 6 seconds (past TTL):")
print(f"  TTL: {r.ttl('flash_sale:item42')}")  # -2 means key doesn't exist
print(f"  Value: {r.get('flash_sale:item42')}")  # None

print("\n💡 TTL=-2 means the key has expired and been removed.")
print("   TTL=-1 means the key exists but has no expiration set.")

Key set with 5-second TTL
  TTL remaining: 5 seconds
  Value: {"discount": "50%", "item": "Laptop"}



After 2 seconds:
  TTL remaining: 3 seconds
  Value: {"discount": "50%", "item": "Laptop"}



After 6 seconds (past TTL):
  TTL: -2
  Value: None

💡 TTL=-2 means the key has expired and been removed.
   TTL=-1 means the key exists but has no expiration set.


## 3️⃣ Write-Through Pattern

In Write-Through, every write goes to both the cache AND the database. This ensures the cache is always up to date.

```
Client writes → Update Cache → Update Database → Return OK
                (synchronous — both must succeed)
```

Trade-off: Writes are slower (two writes instead of one), but reads are always fresh.

In [5]:
r.flushdb()

def write_product_through(product_id, data):
    """Write-Through: update both cache and database."""
    cache_key = f"product:{product_id}"
    
    # Step 1: Write to cache
    r.set(cache_key, json.dumps(data), ex=300)
    print(f"  📝 Written to cache: {cache_key}")
    
    # Step 2: Write to database
    db_write(cache_key, data)
    print(f"  💾 Written to database: {cache_key}")

def read_product_through(product_id):
    """Read from cache (always fresh due to write-through)."""
    cache_key = f"product:{product_id}"
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached)
    # Fall back to DB if cache was evicted
    data = db_read(cache_key)
    if data:
        r.set(cache_key, json.dumps(data), ex=300)
    return data

# Write a product
print("Writing product:")
write_product_through(1, {"id": 1, "name": "Laptop Pro", "price": 1299.99, "stock": 25})

# Read it back — always hits cache
print("\nReading product:")
product = read_product_through(1)
print(f"  ⚡ Got: {product}")

# Update the price
print("\nUpdating price:")
product["price"] = 1199.99
write_product_through(1, product)

# Read again — cache has the latest value
print("\nReading after update:")
product = read_product_through(1)
print(f"  ⚡ Got: {product}")

print("\n💡 Write-Through keeps cache and DB in sync, but writes are slower.")
print("   Use when read freshness is critical (e.g., inventory counts).")

Writing product:
  📝 Written to cache: product:1
  💾 Written to database: product:1

Reading product:
  ⚡ Got: {'id': 1, 'name': 'Laptop Pro', 'price': 1299.99, 'stock': 25}

Updating price:
  📝 Written to cache: product:1
  💾 Written to database: product:1

Reading after update:
  ⚡ Got: {'id': 1, 'name': 'Laptop Pro', 'price': 1199.99, 'stock': 25}

💡 Write-Through keeps cache and DB in sync, but writes are slower.
   Use when read freshness is critical (e.g., inventory counts).


### Cache-Aside vs Write-Through

| Aspect | Cache-Aside | Write-Through |
|--------|------------|---------------|
| Write path | App writes to DB only | App writes to cache + DB |
| Cache freshness | May serve stale data | Always fresh |
| Write latency | Fast (1 write) | Slower (2 writes) |
| Complexity | Simple | Medium |
| Best for | Read-heavy, tolerates staleness | Read-heavy, needs freshness |

## 4️⃣ Redis as Primary: Distributed Lock

Sometimes Redis IS the database, not just a cache. A distributed lock is a perfect example — the lock state lives only in Redis.

```
Worker 1: INCR lock:ticket_42 → returns 1 → "I got it!" → process → DEL
Worker 2: INCR lock:ticket_42 → returns 2 → "Someone else has it" → wait
```

The TTL ensures the lock is released even if a worker crashes.

In [6]:
r.flushdb()

class RedisLock:
    """A simple distributed lock using Redis INCR + TTL."""
    
    def __init__(self, redis_client, lock_name, ttl_seconds=10):
        self.r = redis_client
        self.lock_key = f"lock:{lock_name}"
        self.ttl = ttl_seconds
    
    def acquire(self, worker_id):
        """Try to acquire the lock. Returns True if successful."""
        # INCR is atomic — only one worker gets value=1
        count = self.r.incr(self.lock_key)
        
        if count == 1:
            # We're first! Set a TTL so the lock auto-releases on crash
            self.r.expire(self.lock_key, self.ttl)
            print(f"  🔒 {worker_id} acquired the lock")
            return True
        else:
            print(f"  ⏳ {worker_id} failed to acquire — lock is held")
            return False
    
    def release(self, worker_id):
        """Release the lock."""
        self.r.delete(self.lock_key)
        print(f"  🔓 {worker_id} released the lock")

# Simulate two workers trying to book the same ticket
lock = RedisLock(r, "ticket_42", ttl_seconds=10)

print("Scenario: Two workers try to book the same ticket\n")

# Worker 1 acquires the lock
got_lock_1 = lock.acquire("Worker-1")
if got_lock_1:
    print("  Worker-1 is processing the booking...")
    time.sleep(0.1)  # Simulate work

# Worker 2 tries while Worker 1 has the lock
got_lock_2 = lock.acquire("Worker-2")
print(f"  Worker-2 got lock? {got_lock_2}")

# Worker 1 releases
lock.release("Worker-1")

# Now Worker 2 can retry
print()
got_lock_2 = lock.acquire("Worker-2")
if got_lock_2:
    print("  Worker-2 is processing the booking...")
    lock.release("Worker-2")

print("\n💡 INCR is atomic — even with millions of concurrent requests,")
print("   exactly one worker gets value=1 and 'wins' the lock.")

Scenario: Two workers try to book the same ticket

  🔒 Worker-1 acquired the lock
  Worker-1 is processing the booking...


  ⏳ Worker-2 failed to acquire — lock is held
  Worker-2 got lock? False
  🔓 Worker-1 released the lock

  🔒 Worker-2 acquired the lock
  Worker-2 is processing the booking...
  🔓 Worker-2 released the lock

💡 INCR is atomic — even with millions of concurrent requests,
   exactly one worker gets value=1 and 'wins' the lock.


## 5️⃣ Redis as Primary: Rate Limiter

A fixed-window rate limiter using INCR + EXPIRE. The count lives in Redis — it IS the source of truth.

```
Request comes in → INCR rate:user42:window_123 → count ≤ limit? → Allow
                                                  count > limit? → Reject (429)
```

In [7]:
r.flushdb()

class RateLimiter:
    """Fixed-window rate limiter using Redis INCR + EXPIRE."""
    
    def __init__(self, redis_client, max_requests=5, window_seconds=10):
        self.r = redis_client
        self.max_requests = max_requests
        self.window_seconds = window_seconds
    
    def is_allowed(self, user_id):
        """Check if a request from this user is allowed."""
        # Create a key that includes the current time window
        window = int(time.time()) // self.window_seconds
        key = f"rate:{user_id}:{window}"
        
        # Atomically increment the counter
        count = self.r.incr(key)
        
        # Set TTL on first request so the key auto-expires
        if count == 1:
            self.r.expire(key, self.window_seconds)
        
        allowed = count <= self.max_requests
        remaining = max(0, self.max_requests - count)
        
        return allowed, count, remaining

# Allow 5 requests per 10-second window
limiter = RateLimiter(r, max_requests=5, window_seconds=10)

print("Rate Limiter: 5 requests per 10-second window\n")

# Simulate 8 requests from the same user
for i in range(8):
    allowed, count, remaining = limiter.is_allowed("user_42")
    status = "✅ Allowed" if allowed else "❌ Rate limited (429)"
    print(f"  Request {i+1}: {status} (count={count}, remaining={remaining})")

print("\n💡 Requests 6-8 are rejected. The window resets after 10 seconds.")
print("   INCR + EXPIRE is atomic and handles concurrent requests safely.")

Rate Limiter: 5 requests per 10-second window

  Request 1: ✅ Allowed (count=1, remaining=4)
  Request 2: ✅ Allowed (count=2, remaining=3)
  Request 3: ✅ Allowed (count=3, remaining=2)
  Request 4: ✅ Allowed (count=4, remaining=1)
  Request 5: ✅ Allowed (count=5, remaining=0)
  Request 6: ❌ Rate limited (429) (count=6, remaining=0)
  Request 7: ❌ Rate limited (429) (count=7, remaining=0)
  Request 8: ❌ Rate limited (429) (count=8, remaining=0)

💡 Requests 6-8 are rejected. The window resets after 10 seconds.
   INCR + EXPIRE is atomic and handles concurrent requests safely.


## 6️⃣ Redis as Primary: Leaderboard

Sorted Sets are perfect for leaderboards. The score data lives in Redis — no need for a separate database for rankings.

In [8]:
r.flushdb()

# Build a real-time gaming leaderboard
def add_score(player, score):
    r.zincrby("leaderboard", score, player)

def get_top_players(count=5):
    return r.zrevrange("leaderboard", 0, count - 1, withscores=True)

def get_player_rank(player):
    rank = r.zrevrank("leaderboard", player)
    score = r.zscore("leaderboard", player)
    return rank + 1 if rank is not None else None, score

# Simulate a game with accumulating scores
players = ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace", "Hank"]
print("🎮 Simulating 20 game rounds...\n")

for round_num in range(20):
    # Each round, a random player earns points
    player = random.choice(players)
    points = random.randint(10, 100)
    add_score(player, points)

# Display the leaderboard
print("🏆 Final Leaderboard:")
print("=" * 40)
for rank, (player, score) in enumerate(get_top_players(len(players)), 1):
    medal = {1: "🥇", 2: "🥈", 3: "🥉"}.get(rank, "  ")
    bar = "█" * int(score / 20)
    print(f"  {medal} #{rank:2d} {player:<10s} {int(score):>4d} pts {bar}")

# Player lookup
print("\nPlayer Lookup:")
for player in ["Alice", "Bob"]:
    rank, score = get_player_rank(player)
    if rank:
        print(f"  {player}: Rank #{rank}, Score: {int(score)}")

total = r.zcard("leaderboard")
print(f"\nTotal players: {total}")
print("\n💡 ZINCRBY, ZREVRANGE, ZREVRANK are all O(log N) — fast even with millions of players!")

🎮 Simulating 20 game rounds...

🏆 Final Leaderboard:
  🥇 # 1 Hank        392 pts ███████████████████
  🥈 # 2 Bob         340 pts █████████████████
  🥉 # 3 Diana       166 pts ████████
     # 4 Eve         156 pts ███████
     # 5 Frank        81 pts ████
     # 6 Grace        77 pts ███
     # 7 Alice        60 pts ███

Player Lookup:
  Alice: Rank #7, Score: 60
  Bob: Rank #2, Score: 340

Total players: 7

💡 ZINCRBY, ZREVRANGE, ZREVRANK are all O(log N) — fast even with millions of players!


## 7️⃣ When to Use Redis as Primary vs Cache

```
Is the data ephemeral (OK to lose)?
├── YES → Redis as primary is fine!
│   Examples: rate limits, sessions, real-time counters, leaderboards
│
└── NO → Redis as cache, durable DB as primary
    Examples: user accounts, orders, payments, inventory

Special cases:
├── Need durability + speed? → Redis with AOF persistence
├── Need strong durability? → AWS MemoryDB (Redis-compatible, disk-backed)
└── Need transactions? → Use PostgreSQL/MySQL as primary
```

| Use Case | Redis Role | Why |
|----------|-----------|-----|
| Page caching | Cache | Data lives in DB, Redis speeds up reads |
| Session storage | Primary | Sessions are ephemeral, speed matters |
| Rate limiting | Primary | Counters are ephemeral, atomicity matters |
| Leaderboards | Primary | Rankings are computable, speed matters |
| Distributed locks | Primary | Lock state is ephemeral by nature |
| User profiles | Cache | Durable data belongs in a database |
| Order history | Cache | Must survive Redis restarts |

## 8️⃣ Bad → Best: A Safer Distributed Lock

The lock above (using `INCR` + `EXPIRE`) is great for teaching the idea, but it has two real bugs you'd hit in production:

1. **Non-atomic acquire.** `INCR` runs first, then `EXPIRE`. If the worker crashes *between* those two commands, the lock has no TTL and stays "held" forever — a permanent deadlock.
2. **Anyone can release.** `DEL lock:foo` removes the lock no matter who is calling. If Worker-A's work runs longer than the TTL, Redis auto-expires its lock, Worker-B grabs it, and then Worker-A's *delayed* `DEL` releases Worker-B's lock by accident.

The production-safe pattern fixes both:

- Use **`SET key value NX EX ttl`** — a single atomic command that sets the key only if it doesn't exist *and* attaches the TTL in one shot.
- Use a **unique token** (e.g. a UUID) as the value — this is "proof of ownership".
- Release with a **Lua script** that checks the token before deleting — atomic compare-and-delete.

This is the building block behind the [Redlock](https://redis.io/docs/latest/develop/use-cases/distributed-locks/) algorithm.


In [9]:
import uuid

r.flushdb()

# Lua script: only delete the key if its value matches our token.
# KEYS[1] = lock key, ARGV[1] = our token.
# Returns 1 if we owned and released it; 0 if someone else now owns it.
RELEASE_LOCK_LUA = '''
if redis.call("GET", KEYS[1]) == ARGV[1] then
    return redis.call("DEL", KEYS[1])
else
    return 0
end
'''

class SafeRedisLock:
    """Production-style lock: atomic SET NX EX + token + Lua release."""

    def __init__(self, redis_client, lock_name, ttl_seconds=10):
        self.r = redis_client
        self.lock_key = f"lock:{lock_name}"
        self.ttl = ttl_seconds
        self.token = None
        # Pre-register the Lua script (Redis caches it by SHA1)
        self._release = self.r.register_script(RELEASE_LOCK_LUA)

    def acquire(self, worker_id):
        token = str(uuid.uuid4())
        # SET NX EX: atomic "set only if not exists, with TTL"
        ok = self.r.set(self.lock_key, token, nx=True, ex=self.ttl)
        if ok:
            self.token = token
            print(f"  [LOCK] {worker_id} acquired (token={token[:8]}...)")
            return True
        print(f"  [WAIT] {worker_id} failed - lock held by another worker")
        return False

    def release(self, worker_id):
        if self.token is None:
            print(f"  [WARN] {worker_id} called release() without holding the lock")
            return False
        # Atomic compare-and-delete via Lua - only releases OUR lock
        deleted = self._release(keys=[self.lock_key], args=[self.token])
        if deleted:
            print(f"  [UNLOCK] {worker_id} released (proven owner)")
        else:
            print(f"  [WARN] {worker_id}'s lock had already expired - someone else owns it now")
        self.token = None
        return bool(deleted)


lock = SafeRedisLock(r, "ticket_42", ttl_seconds=10)

print("Scenario 1: Normal acquire / release\n")
lock.acquire("Worker-1")
lock.release("Worker-1")

print("\nScenario 2: TTL expired, then late release tries to free a NEW owner's lock")
lock_a = SafeRedisLock(r, "ticket_42", ttl_seconds=1)
lock_b = SafeRedisLock(r, "ticket_42", ttl_seconds=10)

lock_a.acquire("Worker-A")
print("  ... Worker-A is slow, sleeping past the TTL ...")
time.sleep(1.2)                         # Worker-A's lock auto-expires
lock_b.acquire("Worker-B")              # Worker-B grabs the now-free lock
lock_a.release("Worker-A")              # Worker-A finally tries to release — must NOT delete B's lock!
# Confirm Worker-B still owns the lock
still_held = r.get("lock:ticket_42")
print(f"  Lock still owned by someone? {still_held is not None} (token in Redis = {still_held[:8] if still_held else None}...)")
lock_b.release("Worker-B")

print("\n[INSIGHT] SET NX EX is atomic, and the Lua-based release is a safe")
print("          'check-and-delete'. Together they prevent the two classic bugs.")

Scenario 1: Normal acquire / release

  [LOCK] Worker-1 acquired (token=21935cca...)
  [UNLOCK] Worker-1 released (proven owner)

Scenario 2: TTL expired, then late release tries to free a NEW owner's lock
  [LOCK] Worker-A acquired (token=338a643a...)
  ... Worker-A is slow, sleeping past the TTL ...


  [LOCK] Worker-B acquired (token=8c9d6d52...)
  [WARN] Worker-A's lock had already expired - someone else owns it now
  Lock still owned by someone? True (token in Redis = 8c9d6d52...)
  [UNLOCK] Worker-B released (proven owner)

[INSIGHT] SET NX EX is atomic, and the Lua-based release is a safe
          'check-and-delete'. Together they prevent the two classic bugs.


## 9️⃣ Bad → Best: Sliding Window Rate Limiter

The fixed-window limiter from section 5 is simple but has a famous flaw: **boundary bursts**. If the window resets at second `10`, a user can fire `max_requests` at `t=9.9` and `max_requests` again at `t=10.0` — effectively **2x the intended rate** in a 200 ms window.

The **sliding window log** uses a Sorted Set whose score is the request timestamp. Each request:
1. Removes timestamps older than `now - window` (`ZREMRANGEBYSCORE`).
2. Adds the current timestamp (`ZADD`).
3. Counts the remaining timestamps (`ZCARD`).
4. Sets a TTL slightly longer than the window so the key auto-cleans up if the user goes idle.

It's a few more commands, but the limit is enforced over *any* moving window — no boundary burst.


In [10]:
r.flushdb()


class FixedWindowLimiter:
    """Restated for clarity: same algorithm as section 5, parameterised on a
    `now` timestamp so we can reproduce the boundary scenario exactly."""

    def __init__(self, redis_client, max_requests, window_seconds):
        self.r = redis_client
        self.max_requests = max_requests
        self.window_seconds = window_seconds

    def is_allowed(self, user_id, now):
        window = int(now) // self.window_seconds
        key = f"fw:{user_id}:{window}"
        count = self.r.incr(key)
        if count == 1:
            self.r.expire(key, self.window_seconds)
        return (count <= self.max_requests, count)


class SlidingWindowLimiter:
    """True sliding window using a Sorted Set of request timestamps."""

    def __init__(self, redis_client, max_requests, window_seconds):
        self.r = redis_client
        self.max_requests = max_requests
        self.window_seconds = window_seconds

    def is_allowed(self, user_id, now):
        key = f"sw:{user_id}"
        now_ms = int(now * 1000)
        cutoff = now_ms - self.window_seconds * 1000

        pipe = self.r.pipeline()
        pipe.zremrangebyscore(key, 0, cutoff)        # drop expired entries
        pipe.zadd(key, {str(now_ms): now_ms})        # record this request
        pipe.zcard(key)                              # count requests in the window
        pipe.expire(key, self.window_seconds + 1)
        _, _, count, _ = pipe.execute()
        return (count <= self.max_requests, count)


# Construct the classic boundary attack DETERMINISTICALLY:
#   - 3 requests right before the window boundary at t = 1000.0
#   - 3 more requests right after  the boundary at t = 1000.1
# Limit = 3 requests per 2-second window.

fixed   = FixedWindowLimiter(r, max_requests=3, window_seconds=2)
sliding = SlidingWindowLimiter(r, max_requests=3, window_seconds=2)

# Window for fixed limiter changes at every even integer second.
pre_times  = [999.90, 999.95, 999.99]   # all in window 999//2 = 499
post_times = [1000.01, 1000.05, 1000.10] # all in window 1000//2 = 500 (NEW window!)

print("Limit = 3 requests / 2-second window\n")
print("--- Fixed window ---")
for t in pre_times:
    ok, c = fixed.is_allowed("alice", t)
    print(f"  t={t:>7.2f}  allowed={ok}  count={c}")
print("  --- crossing the boundary at t=1000.00 ---")
for t in post_times:
    ok, c = fixed.is_allowed("alice", t)
    print(f"  t={t:>7.2f}  allowed={ok}  count={c}")
print("  >>> 6 requests within 0.20 seconds were ALL allowed - 2x burst!\n")

# Reset and try the same attack against the sliding window
r.flushdb()
sliding = SlidingWindowLimiter(r, max_requests=3, window_seconds=2)

print("--- Sliding window ---")
for t in pre_times:
    ok, c = sliding.is_allowed("alice", t)
    print(f"  t={t:>7.2f}  allowed={ok}  count={c}")
print("  --- 'crossing the boundary' (no boundary exists in a sliding window) ---")
for t in post_times:
    ok, c = sliding.is_allowed("alice", t)
    print(f"  t={t:>7.2f}  allowed={ok}  count={c}")
print("  >>> Sliding window enforces the limit over ANY 2-second slice.")

print("\n[INSIGHT] Sliding window costs O(log N) per request and a bit more memory,")
print("          but eliminates boundary bursts. Use it for any user-facing limit")
print("          where 'fairness' matters (login attempts, signup throttling, ...).")


Limit = 3 requests / 2-second window

--- Fixed window ---
  t= 999.90  allowed=True  count=1
  t= 999.95  allowed=True  count=2
  t= 999.99  allowed=True  count=3
  --- crossing the boundary at t=1000.00 ---
  t=1000.01  allowed=True  count=1
  t=1000.05  allowed=True  count=2
  t=1000.10  allowed=True  count=3
  >>> 6 requests within 0.20 seconds were ALL allowed - 2x burst!

--- Sliding window ---
  t= 999.90  allowed=True  count=1
  t= 999.95  allowed=True  count=2
  t= 999.99  allowed=True  count=3
  --- 'crossing the boundary' (no boundary exists in a sliding window) ---
  t=1000.01  allowed=False  count=4
  t=1000.05  allowed=False  count=5
  t=1000.10  allowed=False  count=6
  >>> Sliding window enforces the limit over ANY 2-second slice.

[INSIGHT] Sliding window costs O(log N) per request and a bit more memory,
          but eliminates boundary bursts. Use it for any user-facing limit
          where 'fairness' matters (login attempts, signup throttling, ...).


## 🔟 Cache Stampede (Thundering Herd) — and how to avoid it

A **cache stampede** is a classic outage pattern: a popular key expires, hundreds of requests miss the cache *at the same moment*, and they all dog-pile the database. The DB melts.

Two simple mitigations cover most cases:

1. **TTL jitter** — randomise the TTL by ±10–20 % so a million keys don't expire at the exact same second.
2. **Single-flight (mutex) recompute** — when a miss occurs, only one worker rebuilds the value; everyone else waits briefly and then re-reads from cache. We use `SET NX EX` (the same primitive as the safe lock) as a short-lived "I'm already recomputing" flag.


In [11]:
r.flushdb()
import random as _random
import threading as _threading

DB_CALLS = {"count": 0}

def expensive_db_lookup(product_id):
    """Pretend this query takes 200ms."""
    DB_CALLS["count"] += 1
    time.sleep(0.2)
    return {"id": product_id, "name": f"Product {product_id}", "price": 9.99}

# ----- BAD: naive cache-aside, no protection -----
def get_naive(pid):
    key = f"naive:{pid}"
    cached = r.get(key)
    if cached:
        return json.loads(cached)
    val = expensive_db_lookup(pid)
    r.set(key, json.dumps(val), ex=10)
    return val

# Simulate a stampede: 50 threads all asking for the same just-expired key
DB_CALLS["count"] = 0
threads = [_threading.Thread(target=get_naive, args=(42,)) for _ in range(50)]
[t.start() for t in threads]
[t.join() for t in threads]
print(f"BAD  : 50 concurrent requests -> {DB_CALLS['count']} DB calls (every miss hit the DB)")

# ----- BEST: single-flight with SET NX EX + TTL jitter -----
def get_singleflight(pid):
    key = f"sf:{pid}"
    lock_key = f"sf-lock:{pid}"

    cached = r.get(key)
    if cached:
        return json.loads(cached)

    # Try to become "the one" who recomputes. NX = only set if missing.
    # The short TTL (5s) is a safety net so a crashed worker can't block forever.
    got_lock = r.set(lock_key, "1", nx=True, ex=5)
    if got_lock:
        try:
            val = expensive_db_lookup(pid)
            ttl = 10 + _random.randint(-2, 2)  # +/- 20% jitter
            r.set(key, json.dumps(val), ex=ttl)
            return val
        finally:
            r.delete(lock_key)
    else:
        # Someone else is recomputing — wait briefly and re-read the cache
        for _ in range(20):
            time.sleep(0.02)
            cached = r.get(key)
            if cached:
                return json.loads(cached)
        # Last resort: just hit the DB ourselves (rare)
        return expensive_db_lookup(pid)

DB_CALLS["count"] = 0
threads = [_threading.Thread(target=get_singleflight, args=(42,)) for _ in range(50)]
[t.start() for t in threads]
[t.join() for t in threads]
print(f"BEST : 50 concurrent requests -> {DB_CALLS['count']} DB call(s) - single-flight worked!")

print("\n[INSIGHT] Cache stampedes are an availability bug, not just a perf one.")
print("          TTL jitter + a tiny SET-NX 'recompute lock' eliminate them.")

BAD  : 50 concurrent requests -> 50 DB calls (every miss hit the DB)


BEST : 50 concurrent requests -> 1 DB call(s) - single-flight worked!

[INSIGHT] Cache stampedes are an availability bug, not just a perf one.
          TTL jitter + a tiny SET-NX 'recompute lock' eliminate them.


## 🧹 Cleanup

In [12]:
r.flushdb()
print("🧹 Cleaned up all keys from this notebook")

🧹 Cleaned up all keys from this notebook


## 📚 Summary

### Key Takeaways

1. **Cache-Aside** is the most common pattern — check cache first, fall back to DB on miss
2. **Write-Through** keeps cache always fresh but adds write latency
3. **TTL** prevents stale data and bounds cache size — always set it!
4. **Distributed locks** use INCR + TTL — exactly one worker gets value=1
5. **Rate limiters** use INCR + EXPIRE — simple, atomic, and concurrent-safe
6. **Leaderboards** use Sorted Sets — O(log N) for all operations
7. **Redis as primary** works for ephemeral data (sessions, counters, locks, rankings)
8. **Redis as cache** is better when data must survive restarts (use a durable DB)

### Next Up

In **Notebook 4**, we'll explore **Cluster and Replication** — how to make Redis highly available with Sentinel, scale with clustering, and handle the hot key problem.